In [ ]:


!pip install torch==2.7.1 torchvision==0.22.1 torchaudio==2.7.1 --index-url https://download.pytorch.org/whl/cu118


In [1]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

2.7.1+cu118
True


In [ ]:
# Install core Hugging Face + PEFT + TRL + Accelerate + Datasets
!pip install -U transformers accelerate datasets peft trl

# Install latest stable bitsandbytes (auto-detects CUDA in Colab)
!pip install --upgrade bitsandbytes

#!pip install tensorboard

!pip install --upgrade trl
# Check GPU & bitsandbytes nn module

In [ ]:
import torch
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version compiled:", torch.version.cuda)
print("Device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")

In [ ]:
import trl
print(trl.__version__)


In [ ]:
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    HfArgumentParser,
    TrainingArguments,
    pipeline,
    logging,
)
from peft import LoraConfig, PeftModel
from trl import SFTTrainer

torch.cuda.empty_cache()

In [ ]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
#!pip install -q datasets
#!huggingface-cli-login

In [ ]:
print("PyTorch version:", torch.__version__)

In [ ]:


# load the dataset
dataset = load_dataset("timdettmers/openassistant-guanaco")

# Shuffle the dataset and slice it
dataset = dataset["train"].shuffle(seed=42).select(range(1000))

# Define a function to transform the data
def transform_conversation(example):
  conversation_text = example['text']
  segments = conversation_text.split('###')

  reformatted_segment = []

  # Itrate over pair of segments
  for i in range(1, len(segments)-1,2):
    human_text = segments[i].strip().replace('Human:','').strip()

    # check if there is a corresponding assistant segment before procesiing
    if i + 1 < len(segments):
      assistant_text = segments[i+1].strip().replace('Assistant:','').strip()

      # Apply the new template
      reformatted_segment.append(f'<|user|>\n{human_text}<|end|>\n<|assistant|>\n{assistant_text}<|end|>\n')
    else:
      # Handle the case  where there is no corresponding assistant segment
      reformatted_segment.append(f'<|user|>\n{human_text}<|end|>\n<|assistant|>\n')

    return{'text': ''.join(reformatted_segment)}

transformed_dataset = dataset.map(transform_conversation)

In [ ]:
# The model that you want to train from the Hugging Face hub

model_name = "microsoft/Phi-3-mini-4k-instruct"

# The instruction dataset to use
dataset_name = "timdettmers/openassistant-guanaco"

# Fine-tuned model name
new_model = "finetunedModel"

################################################################################
# QLoRA parameters
################################################################################

# LoRA Attention dimension
lora_r = 64

# Alpha parameter for LoRA scalling
lora_alpha = 16

# Dropout probability for LoRA layers
lora_dropout = 0.1

################################################################################
# bitsAndBytes parameter
################################################################################

# Activate 4-bit precision base model loading
use_4bit = True

# Compute dtype for 4-bit base models
bnb_4bit_compute_dtype = "float16"

# Quantization type (fp4 or nf4)
bnb_4bit_quant_type = "nf4"

# Activarte nested quantization for 4-bit base models (double quantization)
use_nested_quant = True

################################################################################
# TrainngArguments parameters
################################################################################

#Output directory where the model predictions and checkpoints will be stored
output_dir = "./results"

# Number of training epochs
num_train_epochs = 1

# Enable fp16/bf16 training (set bf16 to True with an A100)
fp16 = False
bf16 = False

# Batch size per GPU for training
per_device_train_batch_size =1

# Batch size per GPU for evaluation
per_device_eval_batch_size = 4

# Number of update steps to accumulate the gradients for
gradient_accumulation_steps = 1

# Enable gradient checkpointing
gradient_checkpointing = True

# Maximum gradient normal (gradient clipping)
max_grad_norm = 0.3

# Initial learining rate (AdamW optimizer)
learning_rate = 2e-4

# weight decay to apply to all layers except bias/LayerNorm weights
weight_decay = 0.001

# Optimezer to use
optim = "paged_adamw_32bit"

# Learning rate schedule
lr_scheduler_type = "cosine"

# Number of training steps (overrides num_train_epochs)
max_steps = -1

# Ration of steps for a linear warmup (from 0 to learning rate)
warmup_ratio = 0.03

# Group sequences into batches with same length
# Saves memory and speeds up training considerably
group_by_length = True



# save checkpoint every X updates steps
save_steps = 0

# Log every X updates steps
logging_steps = 15

################################################################################
# SFT parameters
################################################################################

# Maximum sequence length to use max_seq_length = None
max_seq_length = None

# Pack multiple short examples in the same input sequence to incease efficiency
packing = False

# Load the entire model on the GPU 0
#device_map =  {"": 0}





In [ ]:
import torch
print(torch.cuda.is_available())

In [ ]:
#!pip install tensorboardX



In [ ]:
# Load Dataset

dataset = load_dataset(dataset_name, split="train")


compute_dtype = getattr(torch, bnb_4bit_compute_dtype)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=use_4bit,
    bnb_4bit_quant_type=bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=use_nested_quant,
    llm_int8_enable_fp32_cpu_offload=True   # ADD THIS
)

if compute_dtype == torch.float16 and use_4bit:
    major, _ = torch.cuda.get_device_capability()
    if major >= 8:
        print("=" * 80)
        print("Your GPU supports bfloat16: accelerate training with bf16=True")
        print("=" * 80)





model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    offload_folder="offload",
    #offload_state_dict=True
)

model.config.use_cache = False
model.config.pretraining_tp = 1


tokenizer = AutoTokenizer.from_pretrained(model_name,trust_remote_code = True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"


# Load LoRA configuration
peft_config = LoraConfig(
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    r=lora_r,
    bias="none",
    task_type="CAUSAL_LM",
)


# Set Train Parameter
training_arguments = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    optim=optim,
    save_steps=save_steps,
    logging_steps=logging_steps,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    fp16=fp16,
    bf16=bf16,
    max_grad_norm=max_grad_norm,
    max_steps=max_steps,
    warmup_ratio=warmup_ratio,
    group_by_length=group_by_length,
    lr_scheduler_type=lr_scheduler_type,
    report_to="tensorboard"
)

# Set Supervised Finetune Parameters

def formatting_func(example):
    return example["text"] # Changed 'text' to 'dataset'


tokenized_dataset = dataset.map(
    lambda e: tokenizer(formatting_func(e), truncation=True, padding="max_length",max_length=128),
    batched=True
)


trainer = SFTTrainer(
    model=model,
    train_dataset=tokenized_dataset,
    args=training_arguments,
    peft_config=peft_config
)

trainer.train()

In [ ]:
# save trained model
trainer.model.save_pretrained(new_model)
tokenizer.save_pretrained(new_model)

## Step 5: Check the plots on tensorboard, as follows

In [ ]:
'''%load_ext tensorboard
%tensorboard --logdir results/runs'''

## Step 6: Use the text generation pipeline to ask questions with the Phi-3-mini chat template.

In [ ]:
# ignore warnings
logging.set_verbosity(logging.CRITICAL)

# Run text generation pipeline with our next model
pipe = pipeline(task ="text-generation", model= model, tokenizer = tokenizer, max_length = 1500)


In [ ]:
prompt = "i am good how about you"
result = pipe(f"<|user|>\n{prompt}<|end|>\n<|assistant|>\n")
print(result[0]['generated_text'])

In [ ]:
'''import gc
import torch

del model   # purana model hata
gc.collect()
torch.cuda.empty_cache()   # GPU memory free'''